# PyGeoFetch — 09: Complete Geospatial Processing Engine

The full processing engine: preprocessing, 17 spectral indices, post-processing,
SAR processing, batch processing, and pipeline templates.

### What you'll learn
- All 11 preprocessing operations
- All 17 spectral indices with formulas and use cases
- All 9 post-processing operations
- All 4 SAR operations
- Parallel batch processing
- Processing pipeline templates
- ProcessingResult metadata fields

In [1]:
from pygeofetch import PyGeoFetch

client = PyGeoFetch(log_level="WARNING")

# Verify all subsystems
subsystems = ["preprocess", "indices", "post", "sar", "batch", "pipeline"]
for s in subsystems:
    assert hasattr(client, s), f"Missing subsystem: {s}"
    print(f"  client.{s}  ✓")
print()
print("All 6 processing subsystems ready")

  client.preprocess  ✓
  client.indices  ✓
  client.post  ✓
  client.sar  ✓
  client.batch  ✓
  client.pipeline  ✓

All 6 processing subsystems ready


## 1. Preprocessing — 11 Operations

In [ ]:
print("Preprocessing operations (client.preprocess.METHOD()):")
print()

operations = [
    ("atmos",        "Atmospheric correction",        "dos1, sen2cor, flaash, 6s, icor"),
    ("topo_correct", "Topographic correction",        "cosine, minnaert, c_correction"),
    ("cloud_mask",   "Cloud masking",                 "scl, fmask, threshold, ndsi"),
    ("cloud_fill",   "Cloud filling",                 "interpolate, nearest"),
    ("clip",         "Geometric clip",                "bbox or geometry GeoJSON"),
    ("reproject",    "CRS reprojection",              "any EPSG or proj string"),
    ("resample",     "Spatial resampling",            "nearest, bilinear, cubic, lanczos"),
    ("pansharpen",   "Pan-sharpening",                "brovey, ihs, gram_schmidt, simple_mean"),
    ("tile",         "Tile for AI/ML",                "tile_size, overlap, min_coverage"),
    ("mosaic",       "Scene mosaicking",              "first, last, min, max"),
    ("composite",    "Temporal compositing",          "median, mean, max, min, best_pixel"),
]

print(f"  {'Method':<16} {'Description':<30} {'Options'}")
print("-" * 75)
for method, desc, opts in operations:
    print(f"  {method:<16} {desc:<30} {opts}")

print()
print("Python API example:")
print("  result = client.preprocess.clip(")
print("      'scene.tif', bbox=(-74.1, 40.6, -73.7, 40.9)")
print("  )")
print("  print(result.success, result.output_path, result.duration_seconds)")

In [ ]:
# ProcessingResult object — returned by ALL processing methods
print("ProcessingResult fields:")
print()
fields = [
    ("success",          "bool",      "True if operation completed without error"),
    ("output_path",      "Path|None", "Path to output file (or dir for tiles)"),
    ("operation",        "str",       "e.g. 'ndvi', 'clip:bbox', 'despeckle:lee'"),
    ("duration_seconds", "float",     "Wall-clock time for the operation"),
    ("metadata",         "dict",      "Operation-specific: masked_pct, tile_count, explained_variance_pct, mean_coherence, water_pct"),
    ("error",            "str|None",  "Error message if success=False"),
]
print(f"  {'Field':<22} {'Type':<12} {'Description'}")
print("-" * 75)
for field, ftype, desc in fields:
    print(f"  {field:<22} {ftype:<12} {desc}")

## 2. Spectral Indices — All 17

In [ ]:
indices = [
    ("ndvi",      "(NIR-Red)/(NIR+Red)",                      "-1 to +1",   "Vegetation health. >0.3 = healthy"),
    ("evi",       "G*(NIR-R)/(NIR+C1*R-C2*B+L)",             "-1 to +1",   "Dense canopy, better than NDVI"),
    ("savi",      "(NIR-R)/(NIR+R+L)*(1+L)",                 "-1 to +1",   "Sparse vegetation, soil correction"),
    ("ndwi",      "(Green-NIR)/(Green+NIR)",                  "-1 to +1",   "Water bodies. >0 = water"),
    ("mndwi",     "(Green-SWIR1)/(Green+SWIR1)",              "-1 to +1",   "Urban water separation"),
    ("ndbi",      "(SWIR1-NIR)/(SWIR1+NIR)",                  "-1 to +1",   "Urban/built-up. >0 = urban"),
    ("ndsi",      "(Green-SWIR1)/(Green+SWIR1)",              "-1 to +1",   "Snow/ice. >0.4 = snow"),
    ("ndmi",      "(NIR-SWIR1)/(NIR+SWIR1)",                  "-1 to +1",   "Canopy water content"),
    ("nbr",       "(NIR-SWIR2)/(NIR+SWIR2)",                  "-1 to +1",   "Pre-fire baseline"),
    ("dnbr",      "NBR_pre - NBR_post",                       "varies",     ">0.66 high severity fire"),
    ("tct",       "Matrix coefficients (S2 or L8)",           "3-band",     "Brightness, Greenness, Wetness"),
    ("pca",       "Eigen decomposition",                      "N-band",     "Dimensionality reduction"),
    ("texture",   "Grey Level Co-occurrence (GLCM)",          "varies",     "Contrast, homogeneity, energy, ASM"),
    ("lst",       "Thermal -> Kelvin/Celsius",                "K / degC",   "2-band: Kelvin and Celsius"),
    ("albedo",    "Liang 2001 narrowband->broadband",         "0 to 1",     "Surface reflectance"),
    ("band_math", "Arbitrary expression on B[i]",             "varies",     "Custom index or ratio"),
    ("stack",     "N bands -> multi-band TIF",                "—",          "Combine single-band rasters"),
]

print(f"  {'Index':<10} {'Range':<10} {'Formula (abbreviated)':<38} {'Primary use'}")
print("-" * 90)
for idx, formula, rng, use in indices:
    print(f"  {idx:<10} {rng:<10} {formula[:36]:<38} {use}")

In [ ]:
# Python API for all indices
print("Spectral index Python API examples:")
print()
api_examples = [
    "ndvi = client.indices.ndvi(red='B04.tif', nir='B08.tif')",
    "evi  = client.indices.evi(blue='B02.tif', red='B04.tif', nir='B08.tif')",
    "savi = client.indices.savi(red='B04.tif', nir='B08.tif', L=0.5)",
    "ndwi = client.indices.ndwi(green='B03.tif', nir='B08.tif')",
    "ndbi = client.indices.ndbi(nir='B08.tif', swir1='B11.tif')",
    "dnbr = client.indices.dnbr(pre_nir='B08_pre.tif', pre_swir2='B12_pre.tif',",
    "                            post_nir='B08_post.tif', post_swir2='B12_post.tif')",
    "tct  = client.indices.tct(blue='B02.tif', green='B03.tif', red='B04.tif',",
    "                           nir='B08.tif', swir1='B11.tif', swir2='B12.tif', sensor='sentinel2')",
    "pca  = client.indices.pca(inputs=['B02.tif','B03.tif','B04.tif','B08.tif'], n_components=3)",
    "# pca.metadata['explained_variance_pct']  ->  [45.2, 28.1, 12.3]",
    "lst  = client.indices.lst('B10.tif', emissivity=0.97, sensor='landsat8')",
    "result = client.indices.band_math(['B04.tif','B08.tif'],",
    "                                   expression='(B[1]-B[0])/(B[1]+B[0]+1e-6)')",
]
for line in api_examples:
    print(f"  {line}")

## 3. Post-Processing — 9 Operations

In [ ]:
print("Post-processing operations (client.post.METHOD()):")
print()
post_ops = [
    ("vectorize",           "Raster -> polygons",   "threshold, min_area, format (geojson/gpkg/shp)"),
    ("smooth",              "Geometry simplify",    "tolerance, method (simplify/buffer)"),
    ("regularize",          "Orthogonalize bldgs",  "min_rotation_angle"),
    ("zonal_stats",         "Zonal statistics",     "count, mean, median, min, max, std, percentiles"),
    ("buffer",              "Geometry buffer",       "distance, cap_style, join_style"),
    ("centroids",           "Extract centroids",    "—"),
    ("add_geometry_metrics","Geometry metrics",     "area_m2, perimeter_m, compactness"),
    ("compress",            "Raster compression",   "lzw, deflate, zstd, packbits"),
    ("cog",                 "Cloud Optimized GTiff","compress, blocksize, overviews"),
]
print(f"  {'Method':<25} {'Description':<25} {'Key Options'}")
print("-" * 80)
for method, desc, opts in post_ops:
    print(f"  {method:<25} {desc:<25} {opts}")

print()
print("Pipeline example:")
print("  vecs  = client.post.vectorize('ndvi.tif', threshold=0.3)")
print("  clean = client.post.smooth(str(vecs.output_path), tolerance=0.5)")
print("  stats = client.post.zonal_stats('ndvi.tif', 'parcels.geojson')")
print("  cog   = client.post.cog('ndvi.tif', compress='deflate', blocksize=512)")

## 4. SAR Processing — 4 Operations

In [ ]:
print("SAR processing operations (client.sar.METHOD()):")
print()
sar_ops = [
    ("despeckle", "Speckle filter",           "lee, enhanced_lee, frost, gamma, boxcar",
     "window (default 7)",           "Filtered amplitude raster"),
    ("calibrate", "Radiometric calibration",  "sigma0, gamma0, beta0",
     "in_db (default True)",         "Backscatter coefficient"),
    ("flood_map", "Flood mapping",            "threshold (default -15 dB)",
     "reference (optional pre-event)","Binary water mask"),
    ("coherence", "InSAR coherence",          "—",
     "window (default 7)",           "Coherence 0-1 raster"),
]
print(f"  {'Method':<12} {'Description':<26} {'Modes/Types':<25} {'Key Param'}")
print("-" * 80)
for method, desc, modes, param, output in sar_ops:
    print(f"  {method:<12} {desc:<26} {modes:<25} {param}")

print()
print("Python API:")
print("  result = client.sar.despeckle('s1c_vv.tif', filter='enhanced_lee', window=7)")
print("  result = client.sar.calibrate('s1c_dn.tif', output_type='sigma0', in_db=True)")
print("  result = client.sar.flood_map('post.tif', threshold=-15.0, reference='pre.tif')")
print("  print(result.metadata['water_pct'])    # fraction of pixels classified as water")
print("  result = client.sar.coherence('slc1.tif', 'slc2.tif', window=7)")
print("  print(result.metadata['mean_coherence'])  # 0.0-1.0")

## 5. Batch Processing

In [ ]:
print("Parallel batch processing:")
print()
print("  # Apply chain to multiple inputs in parallel")
print("  results = client.batch_process(")
print("      inputs=['scene1.tif', 'scene2.tif', 'scene3.tif'],")
print("      chain=[")
print("          ('atmos',     {'method': 'dos1'}),")
print("          ('clip',      {'bbox': (-74.1, 40.6, -73.7, 40.9)}),")
print("          ('ndvi',      {}),")
print("          ('cog',       {'compress': 'deflate'}),")
print("      ],")
print("      output_dir='./batch_processed/',")
print("      parallel=4,")
print("  )")
print("  succeeded = sum(1 for r in results if r.success)")
print()
print("  # Custom function per scene")
print("  def my_pipeline(inp, out_dir, threshold=0.3):")
print("      ndvi = client.indices.ndvi(red=inp, nir=inp)")
print("      return client.post.vectorize(str(ndvi.output_path), threshold=threshold)")
print()
print("  results = client.batch.apply(my_pipeline, inputs=['s1.tif','s2.tif'])")

## 6. End-to-End Pipeline — All 41 Steps

In [ ]:
# Build the full 41-step pipeline
pl = (
    client.pipeline("complete-41")
    # ── Preprocessing (11) ──
    .atmos(method="dos1")
    .cloud_mask(method="scl")
    .cloud_fill()
    .clip(bbox=(-74.1, 40.6, -73.7, 40.9))
    .reproject(crs="EPSG:4326")
    .resample(resolution=10)
    .pansharpen(pan="pan.tif", ms="ms.tif")
    .topo_correct(dem="dem.tif")
    .mosaic()
    .composite(method="median")
    .tile(tile_size=512, overlap=64)
    # ── Spectral Indices (17) ──
    .ndvi(red="B04.tif", nir="B08.tif")
    .evi(blue="B02.tif", red="B04.tif", nir="B08.tif")
    .savi(red="B04.tif", nir="B08.tif", L=0.5)
    .ndwi(green="B03.tif", nir="B08.tif")
    .mndwi(green="B03.tif", swir1="B11.tif")
    .ndbi(nir="B08.tif", swir1="B11.tif")
    .ndsi(green="B03.tif", swir1="B11.tif")
    .ndmi(nir="B08.tif", swir1="B11.tif")
    .nbr(nir="B08.tif", swir2="B12.tif")
    .dnbr(pre_nir="B08.tif", pre_swir2="B12.tif", post_nir="B08p.tif", post_swir2="B12p.tif")
    .tct(blue="B02.tif", green="B03.tif", red="B04.tif", nir="B08.tif", swir1="B11.tif", swir2="B12.tif")
    .pca(inputs=["B02.tif","B03.tif","B04.tif","B08.tif"], n_components=3)
    .texture(window=7)
    .lst(emissivity=0.97, sensor="landsat8")
    .albedo()
    .band_math(expression="(B[1]-B[0])/(B[1]+B[0]+1e-6)")
    .stack()
    # ── Post-Processing (9) ──
    .vectorize(threshold=0.3)
    .smooth(tolerance=0.5)
    .regularize()
    .zonal_stats()
    .buffer(distance=15)
    .centroids()
    .add_geometry_metrics()
    .compress(method="lzw")
    .cog(compress="deflate")
    # ── SAR (4) ──
    .despeckle(filter="enhanced_lee", window=7)
    .calibrate(output_type="sigma0", in_db=True)
    .flood_map(threshold=-15.0)
    .coherence(window=7)
)

assert len(pl._steps) == 41
print(f"Pipeline: {pl.name!r}")
print(f"Steps: {len(pl._steps)} (all 41 verified)")
print()
groups = [
    ("Preprocessing", pl._steps[0:11]),
    ("Spectral Indices", pl._steps[11:28]),
    ("Post-Processing", pl._steps[28:37]),
    ("SAR", pl._steps[37:41]),
]
for group, steps in groups:
    print(f"  {group} ({len(steps)}): {', '.join(s.name for s in steps)}")

## 7. Logging Format Verification

In [2]:
from pygeofetch.core.logging import (
    _PGFFormatter,
    _redact,
    _render_progress_bar,
    configure_logging,
    get_logger,
)

# Spec-required format: HH:MM:SS LEVL [   module] message
configure_logging("INFO")
logger = get_logger("search")

# Show the spec format
formatter = _PGFFormatter(use_colour=False, show_module=True)
import logging

record = logging.LogRecord(
    name="pygeofetch.core.search",
    level=logging.INFO,
    pathname="", lineno=0,
    msg="Searching %d provider(s) for %s scenes",
    args=(2, "Sentinel-1"), exc_info=None,
)
print("Spec log format:")
print(f"  {formatter.format(record)}")

# Progress bar
bar = _render_progress_bar(
    completed=3, total=8,
    filename="S1C_IW_GRDH_1SDV_20260601.zip",
    bytes_done=2_100_000_000,
    bytes_total=4_100_000_000,
    speed_bps=18_400_000,
)
print()
print("Progress bar (starts with backslash-r):")
print(f"  {bar.strip()}")

# Credential redaction
msg = "Connecting with api_key=PL-abc123-secret to planet"
redacted = _redact(msg)
print()
print(f"Original:  {msg}")
print(f"Redacted:  {redacted}")
assert "PL-abc123-secret" not in redacted
print("Credential redaction: PASS")

Spec log format:
  23:04:04 INFO [      search] Searching 2 provider(s) for Sentinel-1 scenes

Progress bar (starts with backslash-r):
  [███████░░░░░░░░░░░░░]  3/8  S1C_IW_GRDH_1SDV_20260601.zip               2.1 GB / 4.1 GB  18.4 MB/s

Original:  Connecting with api_key=PL-abc123-secret to planet
Redacted:  Connecting with api_key=***REDACTED*** to planet
Credential redaction: PASS


---
## Summary

| Component | Count | Methods |
|---|---|---|
| Preprocessing | 11 | atmos, topo_correct, cloud_mask, cloud_fill, clip, reproject, resample, pansharpen, tile, mosaic, composite |
| Spectral Indices | 17 | ndvi, evi, savi, ndwi, mndwi, ndbi, ndsi, ndmi, nbr, dnbr, tct, pca, texture, lst, albedo, band_math, stack |
| Post-Processing | 9 | vectorize, smooth, regularize, zonal_stats, buffer, centroids, add_geometry_metrics, compress, cog |
| SAR Processing | 4 | despeckle, calibrate, flood_map, coherence |
| **Total** | **41** | All chainable via Python builder or YAML |

All bug fixes (BUG 1-4) and new capabilities (CAP 1-3) integrated into the engine.